# Gerador do dataset sintético

Gera `dataset_estatistica.csv` (200 clientes de e-commerce fictícios), usado em todos os notebooks do repositório.

| Variável | Distribuição | Por quê |
|---|---|---|
| `renda_mensal` | Lognormal | assimetria à direita e outliers |
| `compras_ano` | Poisson | contagem discreta |
| `nota_satisfacao` | Uniforme(1, 5) | variável limitada |
| `categoria_frequente` | Categórica | moda e variáveis qualitativas |
| `peso_avaliacoes` | Inteiro aleatório | médias ponderadas |

Valores ausentes foram inseridos de propósito em `renda_mensal` e `nota_satisfacao`.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

# Pasta de saída: funciona se o notebook for aberto na raiz do repo ou dentro de data/
DATA_DIR = Path.cwd() if Path.cwd().name == "data" else Path("data")
DATA_DIR.mkdir(exist_ok=True)

# Semente para reprodutibilidade
np.random.seed(42)
n_samples = 200

# 1. Dados cadastrais e categóricos
cliente_id = range(1001, 1001 + n_samples)
idade = np.random.randint(18, 70, size=n_samples)
categoria_frequente = np.random.choice(
    ["Eletrônicos", "Roupas", "Livros", "Alimentos"], size=n_samples, p=[0.3, 0.4, 0.2, 0.1]
)

# 2. Renda mensal: assimetria à direita (lognormal) + outliers naturais
renda_mensal = np.round(np.random.lognormal(mean=8.3, sigma=0.6, size=n_samples), 2)

# 3. Compras no último ano (contagem discreta)
compras_ano = np.random.poisson(lam=5, size=n_samples)

# 4. Nota de satisfação (1.0 a 5.0) e peso (volume de avaliações)
nota_satisfacao = np.round(np.random.uniform(1.0, 5.0, size=n_samples), 1)
peso_avaliacoes = np.random.randint(1, 50, size=n_samples)

df = pd.DataFrame({
    "cliente_id": cliente_id,
    "idade": idade,
    "categoria_frequente": categoria_frequente,
    "renda_mensal": renda_mensal,
    "compras_ano": compras_ano,
    "nota_satisfacao": nota_satisfacao,
    "peso_avaliacoes": peso_avaliacoes,
})

# 5. Valores ausentes intencionais (~5% na renda, ~8% na nota)
mask_renda = np.random.choice([True, False], size=n_samples, p=[0.05, 0.95])
mask_nota = np.random.choice([True, False], size=n_samples, p=[0.08, 0.92])
df.loc[mask_renda, "renda_mensal"] = np.nan
df.loc[mask_nota, "nota_satisfacao"] = np.nan   # (antes: "nota_satisfação" criava uma coluna nova)

# 6. Salvar
caminho_csv = DATA_DIR / "dataset_estatistica.csv"
df.to_csv(caminho_csv, index=False)
print(f"Dataset criado em: {caminho_csv}")

# 7. Verificação de sanidade
assert df.shape == (200, 7), df.shape
assert "nota_satisfação" not in df.columns
print(df.isna().sum())
df.head()


Dataset criado em: C:\Users\Lased\Documents\Estatisitica_Probabilidade\data\dataset_estatistica.csv
cliente_id              0
idade                   0
categoria_frequente     0
renda_mensal           11
compras_ano             0
nota_satisfacao        16
peso_avaliacoes         0
dtype: int64


,cliente_id,idade,categoria_frequente,renda_mensal,compras_ano,nota_satisfacao,peso_avaliacoes
0,1001,56,Eletrônicos,4238.39,7,1.1,22
1,1002,69,Roupas,2819.37,8,3.0,28
2,1003,46,Roupas,2675.78,2,2.6,7
3,1004,32,Roupas,3435.63,5,2.8,35
4,1005,60,Eletrônicos,3400.26,4,5.0,28
